<a href="https://colab.research.google.com/github/irungus/tree-species-ml-pipeline/blob/main/notebooks/01_data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install earthengine-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.0/478.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 107.3 MB/s eta 0:00:00


In [14]:
import pandas as pd # load and preprocess data
import os #manage file paths and directories.
import requests # sending HTTP requests to interact with web APIs or download content from the internet.
import io #Provides tools for working with I/O streams.
import matplotlib.pyplot as plt # Data visualization
import seaborn as sns# Data visualization
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import matplotlib.pyplot as plt
import ee

In [15]:
url = "https://raw.githubusercontent.com/irungus/tree-species-ml-pipeline/main/data/raw/Data.csv"

import pandas as pd
df = pd.read_csv(url)

print(df.head())

    uid     species  latitude  longitude
0  6042      Acacia -4.537429  39.139728
1  5988  Eucalyptus -4.522492  39.183680
2  5985  Eucalyptus -4.522487  39.183677
3  5982  Eucalyptus -4.522480  39.183657
4  5932  Eucalyptus -4.472281  39.189929


In [16]:
Columns = [
    "longitude", "latitude", "species"
]

## FEATURE EXTRACTION (GEE + Python)

We’ll extract:

### 🌱 Vegetation Indices (Sentinel-2)
- NDVI  
- EVI  
- SAVI  

### 📡 SAR Features (Sentinel-1)
- VV  
- VH  
- VV/VH ratio  

### 🌡 Climate
- Temperature (mean)  
- Precipitation (mean)  

### ⛰ Terrain
- Elevation  
- Slope  

In [20]:
ee.Authenticate()
ee.Initialize(project='agfkenya')

def extract_features(lat, lon):
    point = ee.Geometry.Point([lon, lat])

    # ---------------------------
    # Sentinel-2 (Optical)
    # ---------------------------
    s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(point)
        .filterDate("2022-01-01", "2022-12-31")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        .select(["B2", "B4", "B8"])  # Ensure consistency
        .median()
    )

    # Vegetation Indices
    ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")

    evi = s2.expression(
        "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
        {
            "NIR": s2.select("B8"),
            "RED": s2.select("B4"),
            "BLUE": s2.select("B2"),
        },
    ).rename("EVI")

    savi = s2.expression(
        "((NIR - RED) / (NIR + RED + 0.5)) * 1.5",
        {
            "NIR": s2.select("B8"),
            "RED": s2.select("B4"),
        },
    ).rename("SAVI")

    # ---------------------------
    # Sentinel-1 (SAR)
    # ---------------------------
    s1 = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(point)
        .filterDate("2022-01-01", "2022-12-31")
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .select(["VV", "VH"])
        .median()
    )

    vv_vh_ratio = s1.select("VV").divide(s1.select("VH")).rename("VV_VH_ratio")

    # ---------------------------
    # Terrain
    # ---------------------------
    dem = ee.Image("USGS/SRTMGL1_003").rename("elevation")
    slope = ee.Terrain.slope(dem).rename("slope")

    # ---------------------------
    # Combine all features
    # ---------------------------
    image = ee.Image.cat([
        ndvi, evi, savi,
        s1.select("VV"),
        s1.select("VH"),
        vv_vh_ratio,
        dem,
        slope
    ])

    # ---------------------------
    # Extract values
    # ---------------------------
    values = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=point,
        scale=30,
        bestEffort=True
    )

    # Safe return
    return values.getInfo()

## Apply to dataset

In [1]:
features = []

for i, row in df.iterrows():
    vals = extract_features(row['latitude'], row['longitude'])
    vals['species'] = row['species']
    features.append(vals)

feature_df = pd.DataFrame(features)

NameError: name 'df' is not defined

In [ ]:
print(feature_df)